# **DATA EXPLORATION AND PREPROCESSING**

---

## **1. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style cho biểu đồ
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11

## **2. Load Raw Data**

Trong giai đoạn đầu của quá trình khám phá dữ liệu (EDA), chúng em tiến hành tải dữ liệu thô (raw data) từ tệp CSV được lưu trữ trong cấu trúc thư mục dự án (`data/raw`).

In [ ]:
energy_path = "../../data/raw/Energy_Use.csv"
df_energy = pd.read_csv(energy_path)

Dữ liệu được đọc từ tệp `Energy_Use.csv` và được khởi tạo dưới dạng cấu trúc bảng lưu vào biến DataFrame `df_energy`.

## **3. Basic Dataset Overview**

### **3.1 Dataset Dimensions**

Bước đầu tiên để nắm bắt quy mô của dữ liệu là kiểm tra chiều không gian (số lượng quan sát và số lượng đặc trưng).

In [ ]:
num_rows_energy, num_cols_energy = df_energy.shape
print(f"Number of rows: {num_rows_energy}")
print(f"Number of columns: {num_cols_energy}")

Bộ dữ liệu **Appliances Energy Prediction** bao gồm *19.735 quan sát* và *29 biến đặc trưng*. Cụ thể, không gian đặc trưng bao gồm 27 biến liên tục (đo lường nhiệt độ, độ ẩm trong nhà/ngoài trời, và các thông số thời tiết khác), 1 biến thời gian (`date`) và 1 biến mục tiêu (`Appliances` - lượng điện năng tiêu thụ). Quy mô và sự đa dạng của các biến số này cung cấp một nền tảng vững chắc để thực hiện các phân tích tương quan đa biến và xây dựng các mô hình Hồi quy (Regression) dự báo mức tiêu thụ năng lượng.

### **3.2 Observational Unit**

Đơn vị quan sát cốt lõi của tập dữ liệu này được xác định như sau:

> **Một bản ghi tương ứng với một chu kỳ đo lường tổng hợp kéo dài 10 phút tại một ngôi nhà duy nhất.**

**Đặc điểm cấu trúc dữ liệu:**
Khác với cấu trúc panel-like data ở nhiều địa điểm, bộ dữ liệu này thuần túy mang cấu trúc *Time-series* tần suất cao. Các tín hiệu từ mạng lưới cảm biến không dây ZigBee trong nhà (thu thập mỗi 3.3 phút) và dữ liệu thời tiết thực tế từ trạm khí tượng đã được đồng bộ hóa và tính trung bình theo chu kỳ 10 phút liên tục trong suốt 4.5 tháng.

### **3.3 Initial Data Glimpse**

Để có cái nhìn ban đầu về cấu trúc và đặc điểm của dataset, chúng em hiển thị một số dòng đầu và cuối để kiểm tra định dạng dữ liệu và mức độ nhất quán:

In [ ]:
# Xem 5 dòng đầu
df_energy.head()

In [ ]:
# Xem 5 dòng cuối
df_energy.tail()

**Summary of Column Types**

In [ ]:
df_energy.dtypes.value_counts()

Kết quả phân tích kiểu dữ liệu cho thấy tập dữ liệu `df_energy` cực kỳ lý tưởng cho các bài toán Học máy định lượng. Có tới 28/29 thuộc tính mang kiểu dữ liệu số (`float64` và `int64`), đại diện cho các thông số đo lường vật lý liên tục từ cảm biến (nhiệt độ, độ ẩm, áp suất, tốc độ gió) và mức tiêu thụ điện năng.

Biến duy nhất có kiểu `object` (chuỗi ký tự) là cột `date`. Từ góc độ tiền xử lý, định dạng chuỗi của cột này chưa thể sử dụng trực tiếp cho các phép toán chuỗi thời gian, do đó yêu cầu bắt buộc là phải ép kiểu (type-casting) cột này về định dạng `datetime` chuẩn của Pandas trước khi tiến hành phân tích sâu hơn.

### **3.4 Temporal Coverage & Spatial Distribution**

In [ ]:
df_energy['date'] = pd.to_datetime(df_energy['date'], format='%d-%m-%Y %H:%M')
print(df_energy['date'].min(), "đến", df_energy['date'].max())

**Temporal Coverage:** Dữ liệu trải dài liên tục từ chiều ngày `11/01/2016` đến chiều ngày `27/05/2016` (khoảng 4.5 tháng). Tần suất lấy mẫu dày đặc (mỗi 10 phút/lần) tạo ra một chuỗi thời gian có độ phân giải cao (high-resolution time-series), rất phù hợp để bắt được các chu kỳ sử dụng điện năng ngắn hạn (theo giờ trong ngày) và trung hạn (theo ngày trong tuần).

**Spatial Distribution:** Khác với bộ dữ liệu khí tượng diện rộng, toàn bộ tập dữ liệu này được thu thập tại một địa điểm duy nhất (một ngôi nhà tiết kiệm năng lượng tại Bỉ). Sự cố định về mặt không gian này loại bỏ được nhiễu loạn do sai biệt địa lý, giúp mô hình tập trung hoàn toàn vào việc học các quy luật nhiệt động lực học trong nhà và tác động của thời tiết bên ngoài lên hành vi tiêu thụ điện của gia đình đó.

## **4. Data Semantics**

Phần này phân tích ngữ nghĩa của dữ liệu nhằm xác định ý nghĩa thực tế của các quan sát và các biến số, từ đó vạch ra chiến lược trích xuất đặc trưng phù hợp cho mô hình Hồi quy.

### **4.1 The meaning of each row**

Mỗi dòng trong bộ dữ liệu **Appliances Energy Prediction** đại diện cho một *bản ghi trạng thái tổng hợp trong khoảng thời gian 10 phút* của ngôi nhà.

Trong 10 phút này, hệ thống ghi nhận:

1. Mức điện năng tiêu thụ thực tế của thiết bị gia dụng và đèn chiếu sáng (qua đồng hồ m-bus).
2. Giá trị trung bình của các thông số môi trường trong nhà (nhiệt độ, độ ẩm) được truyền về từ mạng lưới cảm biến không dây ZigBee.
3. Điều kiện thời tiết ngoài trời tại cùng thời điểm đó được nội suy từ trạm khí tượng gần nhất.

**Tính chất cốt lõi:** Dữ liệu mang cấu trúc *Chuỗi thời gian đa biến (Multivariate Time-series Data)*.

**Lưu ý:** Quán tính nhiệt của ngôi nhà khiến nhiệt độ ở phút thứ $t$ phụ thuộc rất mạnh vào phút thứ $t-1$. Tương tự, hành vi bật/tắt thiết bị của con người cũng có tính kế thừa thời gian. Do đó, trong bước mô hình hóa hoặc chia tập Train/Test, ta tuyệt đối *không được dùng random shuffle* dữ liệu mà phải duy trì trật tự thời gian (ví dụ: dùng TimeSeriesSplit) để tránh hiện tượng rò rỉ dữ liệu.

### **4.2 The meaning of each column**

In [ ]:
df_energy.info()

Không gian đặc trưng gồm 29 biến được phân loại thành 5 nhóm ngữ nghĩa chính dựa trên nguồn gốc sinh ra dữ liệu:

**1. Biến thời gian (Temporal Feature):**
* `date`: Thời điểm ghi nhận (YYYY-MM-DD HH:MM:SS). Từ biến này, ta có thể trích xuất thêm các đặc trưng chu kỳ ẩn như giờ trong ngày, ngày trong tuần, tháng trong năm.


**2. Biến mục tiêu và năng lượng (Target & Energy Features):**
* `Appliances`: *(Biến mục tiêu - Target)* Lượng điện năng tiêu thụ của các thiết bị gia dụng chính (đơn vị: Wh). Đây là đại lượng liên tục cần dự báo.
* `lights`: Lượng điện tiêu thụ của các thiết bị chiếu sáng (Wh). Mặc dù là năng lượng, biến này có thể được dùng như một đặc trưng dự báo vì việc bật đèn thường là chỉ báo trực tiếp cho sự hiện diện của con người trong nhà, từ đó kéo theo việc sử dụng các thiết bị khác.

**3. Nhóm biến Môi trường trong nhà (Indoor Thermodynamic Features):**
Bao gồm 18 biến đo lường từ 9 khu vực khác nhau trong nhà. Sự phân chia chi tiết này cho phép đánh giá mức độ truyền nhiệt giữa các phòng:
* *Khu vực sinh hoạt chính:* `T1`/`RH_1` (Bếp), `T2`/`RH_2` (Phòng khách).
* *Khu vực phụ trợ/Sinh hoạt riêng:* `T3`/`RH_3` (Phòng giặt), `T4`/`RH_4` (Phòng làm việc), `T5`/`RH_5` (Phòng tắm), `T7`/`RH_7` (Phòng ủi đồ), `T8`/`RH_8` (Phòng ngủ 2), `T9`/`RH_9` (Phòng ngủ chính).
* *(Dự báo nguy cơ)*: Các biến nhiệt độ trong nhà khả năng cao sẽ xảy ra hiện tượng đa cộng tuyến (Multicollinearity) vì nhiệt độ các phòng thường tăng giảm cùng nhau.

**4. Nhóm biến Thời tiết ngoài trời (Outdoor Meteorological Features):**
Bao gồm các yếu tố tác động trực tiếp đến lớp vỏ công trình và nhu cầu làm mát/sưởi ấm:
* `T6`, `RH_6`: Nhiệt độ và độ ẩm ngay bên ngoài mặt Bắc của ngôi nhà.
* Dữ liệu từ trạm khí tượng: `T_out` (Nhiệt độ môi trường), `RH_out` (Độ ẩm), `Press_mm_hg` (Áp suất), `Windspeed` (Tốc độ gió), `Visibility` (Tầm nhìn), `Tdewpoint` (Nhiệt độ đọng sương).

**5. Nhóm biến Kiểm định (Random Variables):**
* `rv1`, `rv2`: Hai biến số ngẫu nhiên hoàn toàn không có ý nghĩa vật lý. Chúng được các nhà nghiên cứu cố tình đưa vào tập dữ liệu để làm mốc tham chiếu khi đánh giá tầm quan trọng của các đặc trưng. Bất kỳ đặc trưng vật lý nào có độ quan trọng thấp hơn hoặc bằng hai biến ngẫu nhiên này đều có thể bị loại bỏ để giảm chiều dữ liệu.

### **4.3 Column Data Types and Compatibility for Analysis**

In [ ]:
dtype_summary = df_energy.dtypes.value_counts().reset_index()
dtype_summary.columns = ['Data Type', 'Count']
dtype_summary['Example Features'] = dtype_summary['Data Type'].apply(
    lambda x: list(df_energy.select_dtypes(include=x).columns[:3])
)

print("Data Type Distribution:")
display(dtype_summary)
print("\nDetailed Feature Types:")
numerical_feats = df_energy.select_dtypes(include=['float64', 'int64']).columns
categorical_feats = df_energy.select_dtypes(include=['object']).columns
print(f" - Numerical Features ({len(numerical_feats)}): {list(numerical_feats)}")
print(f" - Categorical Features ({len(categorical_feats)}): {list(categorical_feats)}")

Kết quả phân tích cho thấy cấu trúc của tập dữ liệu `df_energy` cực kỳ lý tưởng và tinh gọn cho bài toán Hồi quy (Regression). Không giống như các tập dữ liệu khí tượng diện rộng thường chứa nhiều thông tin phân loại, toàn bộ các đặc trưng dự báo ở đây đều đã ở dạng định lượng.

**Nhóm dữ liệu định lượng (Numerical - float64 & int64):** Chiếm thế áp đảo với 28 biến. Nhóm này trực tiếp biểu diễn các đại lượng vật lý liên tục như điện năng (Wh), nhiệt độ (°C), độ ẩm (%), áp suất (mm Hg) và tốc độ gió (m/s).
* *Định hướng xử lý:* Nhờ bản chất là dữ liệu số, các thuật toán Học máy có thể tính toán trực tiếp mà không cần qua bước mã hóa (Encoding). Tuy nhiên, do các biến này có thang đo (scale) hoàn toàn khác nhau (ví dụ: Áp suất ~700, Nhiệt độ ~20), một bước tiền xử lý *bắt buộc* là phải thực hiện *Chuẩn hóa dữ liệu (Standardization/Min-Max Scaling)* để đảm bảo các mô hình nhạy cảm với khoảng cách (như SVM, KNN, hoặc Neural Networks) có thể hội tụ và hoạt động chính xác.

**Nhóm dữ liệu phân loại (Categorical - object):** Hoàn toàn vắng mặt (0 biến). Việc không có biến chuỗi/phân loại giúp loại bỏ hoàn toàn rủi ro bùng nổ số chiều dữ liệu (Curse of Dimensionality) thường gặp khi phải áp dụng kỹ thuật One-Hot Encoding.

**Nhóm dữ liệu thời gian (Datetime - datetime64[ns]):** Biến `date` duy nhất đã được ép kiểu thành công ở bước trước.
* *Định hướng xử lý:* Không thể đưa trực tiếp chuỗi thời gian nguyên bản vào hầu hết các mô hình Hồi quy. Cần áp dụng kỹ thuật *Feature Engineering* để phân rã cột `date` này thành các đặc trưng phái sinh mang ý nghĩa chu kỳ sinh hoạt của con người, cụ thể như: `Hour_of_day` (giờ trong ngày), `Day_of_week` (ngày trong tuần), hoặc các cờ nhị phân `is_weekend` (là ngày cuối tuần) và `is_business_hour` (trong giờ hành chính).

## **5. Descriptive Statistics**

Thống kê mô tả cung cấp nền tảng định lượng để hiểu xu hướng trung tâm, độ phân tán và đặc điểm phân bố của từng đặc trưng. Chúng ta sẽ tiến hành phân tích thống kê đa tầng:
- **5.1 Overall Statistical Summary:** Thống kê tổng quan trên tất cả các đặc trưng số
- **5.2 Grouped Analysis:** Thống kê theo nhóm ngữ nghĩa đặc trưng

### **5.1 Overall Statistical Summary**

Chúng ta bắt đầu với tổng quan thống kê toàn diện về tất cả các đặc trưng số. Phương thức `describe()` cung cấp bộ 5 số tóm tắt (min, Q1, median, Q3, max) cùng với mean và standard deviation.

In [ ]:
# Calculate descriptive statistics for all numerical features
desc_stats = df_energy.select_dtypes(include='number').describe()

# Display with color gradients for visual pattern recognition
display(desc_stats.T.style
        .format(precision=2)
        .background_gradient(cmap='YlOrRd', subset=['mean', 'std'])
        .background_gradient(cmap='Blues', subset=['min', 'max'])
        .set_caption('Overall Descriptive Statistics - All Numerical Features'))

**Nhận xét chính từ thống kê tổng quan:**

Từ bảng thống kê mô tả toàn diện ở trên, một số quy luật quan trọng được phát hiện:

**1. Sự không đồng nhất về thang đo:**
- `Press_mm_hg` hoạt động trong khoảng 730-770 (thang đo áp suất khí quyển)
- `Appliances` dao động từ 10-1080 Wh (thang đo tiêu thụ năng lượng)
- Nhiệt độ trong nhà (`T1`-`T9`) tập trung quanh 20°C với độ lệch chuẩn hẹp (~1-2°C)
- Các thuật toán dựa trên khoảng cách (KNN, SVM, Neural Networks) sẽ bị chi phối bởi các đặc trưng có độ lớn cao trừ khi chúng ta áp dụng chuẩn hóa

**2. Đặc điểm biến mục tiêu (`Appliances`):**
- Mean ≈ 97.69 Wh với std ≈ 102.52 Wh => Hệ số biến thiên (CV) > 100%
- Độ biến động cực cao này cho thấy tính ngẫu nhiên mạnh trong tiêu thụ năng lượng hộ gia đình
- Khoảng cách lớn giữa Q3 (100 Wh) và max (1080 Wh) chỉ ra sự hiện diện của các ngoại lai tiêu thụ cao
- Có thể yêu cầu các kỹ thuật hồi quy bền vững (robust regression) hoặc xử lý ngoại lai

**3. Tính ổn định nhiệt độ trong nhà:**
- Tất cả các cảm biến trong nhà (`T1`-`T9`) đều cho thấy thống kê tương tự đáng chú ý (mean ~20-21°C, std ~1-2°C)
- Sự đồng nhất này gợi ý khả năng liên kết nhiệt mạnh giữa các phòng
- Nguy cơ gây ra hiện tượng đa cộng tuyến (multicollinearity). Có thể dùng PCA (Principal Component Analysis) hoặc lựa chọn đặc trưng có thể cần thiết

**4. Độ biến động thời tiết ngoài trời:**
- `T_out` dao động từ -5.99°C đến 26.11°C (biến đổi theo mùa)
- `Windspeed` và `Visibility` cho thấy CV cao, chỉ ra sự đa dạng của các sự kiện thời tiết
- Các đặc trưng này mang tín hiệu dự báo mạnh cho nhu cầu sưởi ấm/làm mát

### **5.2 Grouped Analysis by Feature Semantics**

Để có được những hiểu biết sâu sắc hơn theo từng lĩnh vực, chúng em phân chia các đặc trưng thành các nhóm ngữ nghĩa dựa trên ý nghĩa vật lý của chúng. Phân tích theo nhóm này tiết lộ:
- **Tính đồng nhất trong nhóm:** Các cảm biến đo cùng một hiện tượng tương tự như thế nào?
- **Tính không đồng nhất giữa các nhóm:** Nhóm đặc trưng nào thể hiện độ biến động nhiều nhất?
- **Hệ số biến thiên (CV%):** Thước đo độ phân tán tương đối độc lập với thang đo, được tính bằng (std/mean × 100). CV cao cho thấy độ biến động tương đối cao.

In [ ]:
# Define semantic feature groups based on physical domain
variable_groups = {
    'Target Variable': ['Appliances'],
    'Energy Consumption': ['lights'],
    'Indoor Temperature': ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9'],
    'Indoor Humidity': ['RH_1', 'RH_2', 'RH_3', 'RH_4', 'RH_5', 'RH_6', 'RH_7', 'RH_8', 'RH_9'],
    'Outdoor Weather': ['T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint'],
    'Random Variables': ['rv1', 'rv2']
}

# Compute extended statistics for each group
for group_name, variables in variable_groups.items():
    print(f"{group_name}")
    
    group_stats = pd.DataFrame({
        'Mean': df_energy[variables].mean(),
        'Std': df_energy[variables].std(),
        'Min': df_energy[variables].min(),
        'Q1': df_energy[variables].quantile(0.25),
        'Median': df_energy[variables].median(),
        'Q3': df_energy[variables].quantile(0.75),
        'Max': df_energy[variables].max(),
        'Range': df_energy[variables].max() - df_energy[variables].min(),
        'IQR': df_energy[variables].quantile(0.75) - df_energy[variables].quantile(0.25),
        'CV(%)': (df_energy[variables].std() / df_energy[variables].mean() * 100).round(2)
    })
    
    display(group_stats.style
            .format(precision=2)
            .background_gradient(cmap='RdYlGn_r', subset=['CV(%)'])
            .set_caption(f'{group_name} - Detailed Statistics'))

**Nhận xét chuyên sâu theo từng nhóm đặc trưng:**

**1. Biến mục tiêu (`Appliances`):**
- **CV% > 100%:** Hệ số biến thiên cực cao này cho thấy mức tiêu thụ năng lượng rất thất thường và phụ thuộc ngữ cảnh
- **IQR = 60 Wh:** Khoảng tứ phân vị tương đối hẹp so với max (1080 Wh), gợi ý rằng hầu hết mức tiêu thụ xảy ra trong một khoảng cơ sở ổn định với các đỉnh cực đoan thỉnh thoảng
- **Hướng mô hình hóa:** 
  - Xem xét log-transformation để nén đuôi phải
  - Hồi quy bền vững (Huber, RANSAC) có thể vượt trội OLS
  - Các mô hình dựa trên cây (Random Forest, XGBoost) xử lý tự nhiên tính không đồng nhất phương sai này

**2. Cảm biến nhiệt độ trong nhà (`T1`-`T9`):**
- **CV% thấp đáng chú ý (< 10%):** Tất cả các cảm biến đều cho thấy sự tập trung chặt chẽ quanh 20-21°C
- **IQR tương tự (~2-3°C):** Chỉ ra quán tính nhiệt đồng nhất trên tất cả các phòng
- **Nguy cơ đa cộng tuyến:** Sự tương đồng thống kê cho thấy các cảm biến này về cơ bản đang đo cùng một trạng thái nhiệt
- **Khuyến nghị:** 
  - Tính toán `T_indoor_avg` như một đặc trưng tổng hợp
  - Sử dụng PCA để trích xuất các thành phần nhiệt chính
  - Áp dụng phân tích VIF (Variance Inflation Factor) trước khi hồi quy tuyến tính

**3. Cảm biến độ ẩm trong nhà (`RH_1`-`RH_9`):**
- **CV% trung bình (15-25%):** Độ biến động tương đối cao hơn nhiệt độ
- **Khoảng rộng hơn (20-60%):** Độ ẩm dao động mạnh hơn nhiệt độ
- **Hệ quả:** Các cảm biến độ ẩm có thể nắm bắt các tín hiệu hành vi chi tiết hơn (ví dụ: nấu ăn, tắm rửa) có tương quan trực tiếp với việc sử dụng thiết bị

**4. Đặc trưng thời tiết ngoài trời:**
- **`T_out` CV% ≈ 80%:** Biến động nhiệt độ theo mùa từ mùa đông (-6°C) đến mùa xuân (26°C)
- **`Press_mm_hg` CV% < 1%:** Áp suất khí quyển cực kỳ ổn định (như mong đợi)
- **`Windspeed` và `Visibility` CV% cao:** Các biến này nắm bắt các sự kiện thời tiết tạm thời (bão, sương mù)
- **Khuyến nghị:** 
  - `Press_mm_hg` có thể có sức mạnh dự báo thấp do biến động tối thiểu
  - `T_out` có khả năng là yếu tố dự báo mạnh cho các thiết bị sưởi ấm/làm mát
  - Xem xét các số hạng tương tác: `T_out × Is_Winter` hoặc `Temp_Diff × Windspeed`

**5. Biến ngẫu nhiên (`rv1`, `rv2`):**
- **Mục đích:** Đây là các biến nhiễu tổng hợp được các tác giả bộ dữ liệu đưa vào như một kiểm tra tính hợp lý
- **Hành vi mong đợi:** Trong một mô hình được điều chuẩn tốt, chúng nên có độ quan trọng đặc trưng gần bằng không
- **Cờ đỏ:** Nếu `rv1` hoặc `rv2` xếp hạng cao trong độ quan trọng đặc trưng, điều đó cho thấy overfitting
- **Cách sử dụng:** Dùng làm ngưỡng cơ sở cho lựa chọn đặc trưng (loại bỏ bất kỳ đặc trưng thực nào có độ quan trọng < rv1/rv2)

## **6. Target Variable Distribution**

Phân tích toàn diện phân phối của biến mục tiêu `Appliances` nhằm hiểu rõ đặc điểm phân phối (shape, skewness, kurtosis), phát hiện outliers và đưa ra khuyến nghị về transformation cũng như lựa chọn thuật toán phù hợp.

### **6.1 Distribution Shape Analysis**

Trước tiên, chúng em tiến hành phân tích hình dạng phân phối của biến mục tiêu thông qua ba góc nhìn trực quan: Histogram kết hợp KDE để quan sát hình dạng tổng thể, Boxplot để phát hiện ngoại lai, và Q-Q Plot để kiểm định tính chuẩn của phân phối.

In [ ]:
# Tạo subplot với 3 biểu đồ - kích thước lớn hơn
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# 1. Histogram + KDE
axes[0].hist(df_energy['Appliances'], bins=50, edgecolor='black', alpha=0.7, color='steelblue', density=True, linewidth=1.2)
df_energy['Appliances'].plot(kind='kde', ax=axes[0], color='red', linewidth=3)
axes[0].set_xlabel('Energy Consumption (Wh)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Density', fontsize=14, fontweight='bold')
axes[0].set_title('Histogram + KDE: Appliances Distribution', fontsize=15, fontweight='bold', pad=15)
axes[0].axvline(df_energy['Appliances'].mean(), color='green', linestyle='--', linewidth=2.5, 
                label=f'Mean = {df_energy["Appliances"].mean():.1f} Wh')
axes[0].axvline(df_energy['Appliances'].median(), color='orange', linestyle='--', linewidth=2.5, 
                label=f'Median = {df_energy["Appliances"].median():.1f} Wh')
axes[0].legend(fontsize=12, loc='upper right', framealpha=0.9)
axes[0].grid(True, alpha=0.3, linewidth=0.8)
axes[0].tick_params(axis='both', labelsize=11)

# 2. Boxplot
bp = axes[1].boxplot(df_energy['Appliances'], vert=True, patch_artist=True, widths=0.5,
                      boxprops=dict(facecolor='lightblue', color='blue', linewidth=2),
                      medianprops=dict(color='red', linewidth=3),
                      whiskerprops=dict(color='blue', linewidth=2),
                      capprops=dict(color='blue', linewidth=2),
                      flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.5))
axes[1].set_ylabel('Energy Consumption (Wh)', fontsize=14, fontweight='bold')
axes[1].set_title('Boxplot: Outlier Detection', fontsize=15, fontweight='bold', pad=15)
axes[1].grid(True, alpha=0.3, axis='y', linewidth=0.8)
axes[1].tick_params(axis='both', labelsize=11)

# Thêm thông tin quartiles với background
q1 = df_energy['Appliances'].quantile(0.25)
q2 = df_energy['Appliances'].median()
q3 = df_energy['Appliances'].quantile(0.75)
iqr = q3 - q1

axes[1].text(1.25, q1, f'Q1 = {q1:.0f} Wh', fontsize=11, va='center', 
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.3))
axes[1].text(1.25, q2, f'Q2 = {q2:.0f} Wh\n(Median)', fontsize=11, va='center', 
             bbox=dict(boxstyle='round,pad=0.5', facecolor='orange', alpha=0.3))
axes[1].text(1.25, q3, f'Q3 = {q3:.0f} Wh', fontsize=11, va='center', 
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.3))
axes[1].text(1.25, q3 + 100, f'IQR = {iqr:.0f} Wh', fontsize=11, va='center', 
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.3))

# 3. Q-Q Plot
stats.probplot(df_energy['Appliances'], dist="norm", plot=axes[2])
axes[2].get_lines()[0].set_markersize(5)
axes[2].get_lines()[0].set_markerfacecolor('blue')
axes[2].get_lines()[0].set_markeredgecolor('darkblue')
axes[2].get_lines()[0].set_alpha(0.6)
axes[2].get_lines()[1].set_linewidth(2.5)
axes[2].get_lines()[1].set_color('red')
axes[2].set_title('Q-Q Plot: Normality Test', fontsize=15, fontweight='bold', pad=15)
axes[2].set_xlabel('Theoretical Quantiles', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Sample Quantiles', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3, linewidth=0.8)
axes[2].tick_params(axis='both', labelsize=11)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
# Tính toán Skewness, Kurtosis và Jarque-Bera test
skewness = df_energy['Appliances'].skew()
kurtosis = df_energy['Appliances'].kurt()
jb_stat, jb_pvalue = stats.jarque_bera(df_energy['Appliances'])

print("Distribution Shape Analysis for Appliances:")
print(f"Skewness                     : {skewness:.4f}")
print(f"Kurtosis                     : {kurtosis:.4f}")
print(f"Jarque-Bera Statistic        : {jb_stat:.2f}")
print(f"Jarque-Bera p-value          : {jb_pvalue:.2e}")

if jb_pvalue < 0.05:
    print("Conclusion: Distribution is NOT NORMAL (p-value < 0.05)")
else:
    print("Conclusion: Distribution is APPROXIMATELY NORMAL (p-value >= 0.05)")

**Nhận xét về hình dạng phân phối:**

Từ ba biểu đồ trực quan và các chỉ số thống kê ở trên, nhóm rút ra các kết luận quan trọng về đặc điểm phân phối của biến mục tiêu `Appliances`:

**1. Phân phối lệch phải nghiêm trọng (Right-skewed Distribution):**
* **Histogram + KDE:** Đường cong mật độ cho thấy phần lớn các quan sát tập trung ở vùng giá trị thấp (dưới 100 Wh), trong khi có một đuôi dài kéo dài về phía phải với các giá trị cực đoan lên tới 1080 Wh. Hình dạng này không đối xứng và rất khác biệt so với phân phối chuẩn hình chuông.
* **Chỉ số Skewness > 0:** Giá trị dương cao xác nhận tính lệch phải. Điều này phản ánh đúng thực tế vật lý: Trong hầu hết thời gian, ngôi nhà chỉ tiêu thụ mức năng lượng nền thấp (tủ lạnh, các thiết bị chờ), nhưng thỉnh thoảng có các đỉnh tiêu thụ cao đột biến khi nhiều thiết bị công suất lớn (lò nướng, máy giặt, máy sấy) hoạt động đồng thời.

**2. Sự chênh lệch giữa Mean và Median:**
* Mean (≈97.7 Wh) cao hơn đáng kể so với Median (≈60 Wh). Khoảng cách này là dấu hiệu điển hình của phân phối lệch phải, trong đó các giá trị ngoại lai ở đuôi phải kéo Mean lên cao nhưng không ảnh hưởng đến Median.
* Điều này cho thấy nếu sử dụng các mô hình nhạy cảm với Mean (như Linear Regression với MSE loss), chúng sẽ bị ảnh hưởng mạnh bởi các outliers. Trong trường hợp này, Median là thước đo xu hướng trung tâm đáng tin cậy hơn.

**3. Vi phạm giả định phân phối chuẩn:**
* **Q-Q Plot:** Các điểm quan sát (màu xanh) lệch khỏi đường lý thuyết (màu đỏ) một cách rõ rệt, đặc biệt ở hai đầu phân phối. Phần đuôi phải cong lên cao cho thấy có nhiều giá trị cực đoan hơn so với phân phối chuẩn.
* **Jarque-Bera Test:** p-value cực nhỏ (< 0.05) bác bỏ mạnh mẽ giả thuyết H₀ rằng dữ liệu tuân theo phân phối chuẩn. Kết quả này có ý nghĩa thống kê cao.

**4. Kurtosis cao (Heavy-tailed):**
* Giá trị Kurtosis dương cao cho thấy phân phối có đuôi nặng hơn phân phối chuẩn, nghĩa là tồn tại nhiều giá trị cực đoan (outliers) hơn mong đợi. Điều này tăng nguy cơ các mô hình dự báo bị sai lệch nếu không xử lý đúng cách.

**5. Hệ quả đối với mô hình hóa:**
* **Linear Regression thuần túy:** Giả định về phân phối chuẩn của residuals có thể bị vi phạm nghiêm trọng. Các hệ số hồi quy ước lượng bằng OLS (Ordinary Least Squares) sẽ không hiệu quả và khoảng tin cậy có thể không chính xác.
* **Cần transformation:** Các phép biến đổi như Log, Square Root hoặc Box-Cox có thể giúp "nắn" phân phối về dạng gần chuẩn hơn, từ đó cải thiện hiệu suất của các mô hình tuyến tính.

### **6.2 Statistical Summary of Target**

Để có cái nhìn định lượng chi tiết hơn về biến mục tiêu, chúng em tính toán một bộ thống kê mô tả mở rộng bao gồm các thước đo xu hướng trung tâm, độ phân tán, và các percentiles quan trọng.

In [ ]:
# Tạo bảng thống kê chi tiết
target_stats = pd.DataFrame({
    'Metric': [
        'Count', 'Mean', 'Median', 'Mode',
        'Std', 'Variance', 'CV(%)',
        'Min', 'Q1 (25%)', 'Q2 (50%)', 'Q3 (75%)', 'Max',
        'Range', 'IQR',
        'Skewness', 'Kurtosis',
        'Percentile 1%', 'Percentile 5%', 'Percentile 10%',
        'Percentile 90%', 'Percentile 95%', 'Percentile 99%'
    ],
    'Value': [
        df_energy['Appliances'].count(),
        df_energy['Appliances'].mean(),
        df_energy['Appliances'].median(),
        df_energy['Appliances'].mode()[0],
        df_energy['Appliances'].std(),
        df_energy['Appliances'].var(),
        (df_energy['Appliances'].std() / df_energy['Appliances'].mean()) * 100,
        df_energy['Appliances'].min(),
        df_energy['Appliances'].quantile(0.25),
        df_energy['Appliances'].quantile(0.50),
        df_energy['Appliances'].quantile(0.75),
        df_energy['Appliances'].max(),
        df_energy['Appliances'].max() - df_energy['Appliances'].min(),
        df_energy['Appliances'].quantile(0.75) - df_energy['Appliances'].quantile(0.25),
        df_energy['Appliances'].skew(),
        df_energy['Appliances'].kurt(),
        df_energy['Appliances'].quantile(0.01),
        df_energy['Appliances'].quantile(0.05),
        df_energy['Appliances'].quantile(0.10),
        df_energy['Appliances'].quantile(0.90),
        df_energy['Appliances'].quantile(0.95),
        df_energy['Appliances'].quantile(0.99)
    ]
})

display(target_stats.style
        .format({'Value': '{:.2f}'})
        .set_caption('Detailed Statistical Summary of Target Variable: Appliances')
        .background_gradient(cmap='YlOrRd', subset=['Value']))

**Phân tích chi tiết các chỉ số thống kê:**

**1. Thước đo xu hướng trung tâm (Central Tendency):**
* **Mean ≈ 97.7 Wh:** Mức tiêu thụ trung bình bị kéo lên cao bởi các giá trị ngoại lai ở đuôi phải. Đây không phải là đại diện tốt cho mức tiêu thụ "điển hình" của ngôi nhà.
* **Median ≈ 60 Wh:** Giá trị trung vị thấp hơn Mean đáng kể, cho thấy 50% thời gian ngôi nhà tiêu thụ dưới 60 Wh. Đây là thước đo đáng tin cậy hơn cho xu hướng trung tâm trong phân phối lệch.
* **Mode:** Giá trị xuất hiện nhiều nhất, thường tương ứng với mức tiêu thụ nền khi chỉ có các thiết bị cơ bản hoạt động.
* **Khoảng cách Mean - Median ≈ 37.7 Wh:** Khoảng cách lớn này là dấu hiệu rõ ràng của phân phối lệch phải. Trong các mô hình dự báo, việc tối ưu hóa theo Mean (MSE loss) sẽ khác biệt đáng kể so với tối ưu hóa theo Median (MAE loss).

**2. Thước đo độ phân tán (Dispersion):**
* **Std ≈ 102.5 Wh:** Độ lệch chuẩn cao cho thấy mức độ biến động lớn trong tiêu thụ năng lượng.
* **CV% > 100%:** Hệ số biến thiên vượt quá 100% là một tín hiệu cảnh báo nghiêm trọng. Điều này có nghĩa là độ lệch chuẩn lớn hơn cả giá trị trung bình, cho thấy tính ngẫu nhiên và không ổn định cực cao trong hành vi tiêu thụ điện.
* **Variance ≈ 10,506:** Phương sai rất lớn phản ánh sự phân tán rộng của dữ liệu quanh Mean.
* **Hệ quả:** Độ biến động cao này làm cho bài toán dự báo trở nên khó khăn. Các mô hình cần phải học được cả các quy luật chu kỳ (giờ trong ngày, ngày trong tuần) lẫn các yếu tố ngẫu nhiên (hành vi con người).

**3. Phân tích Quartiles và Range:**
* **Min = 10 Wh:** Mức tiêu thụ tối thiểu, có thể tương ứng với trạng thái "ngủ" của ngôi nhà (chỉ có tủ lạnh và một vài thiết bị chờ).
* **Q1 = 50 Wh, Q3 = 100 Wh:** Khoảng tứ phân vị (IQR = 50 Wh) tương đối hẹp, cho thấy 50% dữ liệu trung tâm nằm trong khoảng 50-100 Wh.
* **Max = 1080 Wh:** Giá trị cực đại cao gấp 10 lần Q3, cho thấy sự hiện diện của các sự kiện tiêu thụ cực đoan (có thể là lúc bật lò nướng, máy giặt, máy sấy, điều hòa cùng lúc).
* **Range = 1070 Wh:** Khoảng biến thiên cực lớn so với IQR (50 Wh), chứng tỏ phần lớn độ biến động đến từ các outliers ở đuôi phải.

**4. Phân tích Percentiles:**
* **Percentile 1 - 10%:** Các giá trị rất thấp (dưới 30 Wh), tương ứng với các khoảng thời gian ngôi nhà gần như không có hoạt động.
* **Percentile 90%:** Khoảng 90% thời gian, mức tiêu thụ dưới một ngưỡng nhất định (cần xem giá trị cụ thể từ output).
* **Percentile 95-99%:** Các giá trị này tăng đột biến, cho thấy 5-1% thời gian còn lại có mức tiêu thụ cực cao. Đây chính là các outliers cần được xử lý cẩn thận.
* **Khoảng cách P90 - P99:** Nếu khoảng cách này lớn, ta có thể cân nhắc áp dụng Winsorization (cắt xén giá trị cực đoan về P95 hoặc P99) thay vì loại bỏ hoàn toàn.

## **7. Correlation & Bivariate Analysis**

Phân tích tương quan giúp chúng em xác định các biến môi trường và thời tiết có ảnh hưởng mạnh nhất đến mức tiêu thụ năng lượng (`Appliances`). Chúng em sẽ sử dụng Heatmap để trực quan hóa ma trận tương quan và Scatter plots để kiểm tra tính chất tuyến tính/phi tuyến của các mối quan hệ quan trọng.

### **7.1 Correlation Matrix Heatmap**

Trước tiên, chúng em tính toán ma trận tương quan Pearson cho tất cả các biến số để định lượng mức độ liên kết tuyến tính giữa chúng. Ma trận này sẽ được trực quan hóa bằng Heatmap với màu sắc thể hiện cường độ tương quan.

In [ ]:
# Tính ma trận tương quan
correlation_matrix = df_energy.drop(columns=['date']).corr()

# Vẽ heatmap với kích thước lớn
plt.figure(figsize=(20, 16))
sns.heatmap(correlation_matrix, 
            annot=True,           # Hiển thị giá trị số
            fmt='.2f',            # Format 2 chữ số thập phân
            cmap='RdBu_r',        # Màu: đỏ (dương), xanh (âm)
            center=0,             # Trung tâm tại 0
            square=True,          # Ô vuông
            linewidths=0.5,       # Đường kẻ giữa các ô
            cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"},
            vmin=-1, vmax=1)      # Giới hạn từ -1 đến 1

plt.title('Correlation Matrix Heatmap: Energy Use Dataset', fontsize=18, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.show()

**Cách đọc Heatmap:**
* **Màu đỏ đậm**: Tương quan dương mạnh (gần +1) => Hai biến tăng/giảm cùng chiều
* **Màu xanh đậm**: Tương quan âm mạnh (gần -1) => Hai biến tăng/giảm ngược chiều
* **Màu trắng**: Không có tương quan (gần 0) => Hai biến độc lập với nhau
* **Đường chéo**: Luôn bằng 1 (mỗi biến tương quan hoàn hảo với chính nó)

In [ ]:
appliances_corr = correlation_matrix['Appliances']

# Vẽ biểu đồ thanh cho correlation với Appliances
fig, ax = plt.subplots(figsize=(12, 10))

# Lấy correlation và loại bỏ Appliances với chính nó
corr_with_target = appliances_corr.drop('Appliances').sort_values()

# Tạo màu: đỏ cho dương, xanh cho âm
colors = ['red' if x > 0 else 'blue' for x in corr_with_target]

corr_with_target.plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xlabel('Correlation Coefficient', fontsize=13, fontweight='bold')
ax.set_ylabel('Features', fontsize=13, fontweight='bold')
ax.set_title('Correlation of All Features with Appliances (Target Variable)', 
             fontsize=14, fontweight='bold', pad=15)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
ax.grid(True, alpha=0.3, axis='x')
ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()
plt.show()

**Nhận xét về các nhóm tương quan:**

**1. Biến có tương quan mạnh nhất với Appliances:**
* **`lights` (Chiếu sáng):** Thường có tương quan dương cao nhất. Khi đèn được bật, đó là dấu hiệu con người đang có mặt trong nhà và có khả năng cao sẽ sử dụng các thiết bị khác (tivi, máy tính, bếp). Đây là một proxy indicator (chỉ báo gián tiếp) rất mạnh cho hoạt động sinh hoạt.
* **Biến ngẫu nhiên `rv1`, `rv2`:** Nếu các biến này có correlation cao, đó là dấu hiệu cảnh báo về overfitting hoặc data leakage. Trong thực tế, chúng nên có correlation gần 0.

**2. Nhóm nhiệt độ trong nhà (T1-T9):**
* **Đa cộng tuyến cao:** Các cảm biến nhiệt độ trong nhà (`T1`, `T2`, `T3`... `T9`) thường có correlation rất cao với nhau (> 0.8), vì nhiệt độ các phòng trong cùng một ngôi nhà có xu hướng tăng giảm đồng bộ.
* **Correlation với Appliances:** Thường yếu hoặc trung bình. Lý do là nhiệt độ trong nhà là kết quả của việc sử dụng thiết bị (điều hòa, sưởi), chứ không phải nguyên nhân trực tiếp.
* **Hệ quả:** Trong mô hình Linear Regression, cần cẩn thận với multicollinearity. Có thể sử dụng PCA hoặc chỉ giữ lại một vài biến đại diện (ví dụ: T_indoor_avg đã tạo ở phần Feature Engineering).

**3. Nhóm độ ẩm trong nhà (RH_1-RH_9):**
* Tương tự như nhiệt độ, các cảm biến độ ẩm cũng có correlation cao với nhau.
* Độ ẩm có thể tăng đột biến khi sử dụng các thiết bị như bếp (nấu ăn), máy giặt, hoặc tắm rửa, có thể có correlation dương nhẹ với Appliances.

**4. Nhóm thời tiết ngoài trời:**
* **`T_out` (Nhiệt độ ngoài trời):** Có thể có correlation âm với Appliances trong mùa đông (nhiệt độ thấp, bật sưởi nhiều hơn) hoặc dương trong mùa hè (nhiệt độ cao, bật điều hòa nhiều hơn). Tuy nhiên, mối quan hệ này có thể phi tuyến (hình chữ U).
* **`Press_mm_hg` (Áp suất):** Thường có correlation rất yếu với Appliances vì áp suất khí quyển thay đổi chậm và không ảnh hưởng trực tiếp đến hành vi sử dụng điện.
* **`Windspeed`, `Visibility`, `Tdewpoint`:** Correlation yếu, vì chúng không tác động trực tiếp đến nhu cầu sử dụng thiết bị gia dụng.

**5. Phát hiện quan trọng:**
* Nếu tất cả các biến đều có correlation yếu (< 0.3) với Appliances, điều này cho thấy mối quan hệ giữa các biến môi trường và tiêu thụ năng lượng là **phi tuyến** hoặc **phụ thuộc vào tương tác phức tạp**.
* Trong trường hợp này, các mô hình phi tuyến (Random Forest, XGBoost, Neural Networks) sẽ vượt trội so với Linear Regression.

### **7.2 Bivariate Analysis: Scatter Plots**

Để hiểu rõ hơn về bản chất của mối quan hệ (tuyến tính hay phi tuyến), chúng em vẽ scatter plots cho các cặp biến có correlation mạnh nhất với biến mục tiêu. Scatter plot giúp phát hiện các pattern mà correlation coefficient không thể nắm bắt được.

In [ ]:
# Chọn top 6 biến có correlation mạnh nhất (trị tuyệt đối)
top_features = appliances_corr.drop('Appliances').abs().sort_values(ascending=False).head(6).index.tolist()

print("Top 6 features with strongest correlation (absolute value):")
for i, feat in enumerate(top_features, 1):
    corr_val = appliances_corr[feat]
    print(f"{i}. {feat:20s} : {corr_val:+.4f}")

In [ ]:
# Vẽ scatter plots cho top 6 biến
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    
    # Scatter plot với alpha để thấy mật độ điểm
    ax.scatter(df_energy[feature], df_energy['Appliances'], 
               alpha=0.3, s=10, color='steelblue', edgecolors='none')
    
    # Thêm regression line
    z = np.polyfit(df_energy[feature], df_energy['Appliances'], 1)
    p = np.poly1d(z)
    ax.plot(df_energy[feature], p(df_energy[feature]), 
            "r--", linewidth=2.5, label=f'Linear fit')
    
    # Thêm lowess smooth line để thấy xu hướng phi tuyến
    sorted_idx = np.argsort(df_energy[feature])
    x_sorted = df_energy[feature].iloc[sorted_idx]
    y_sorted = df_energy['Appliances'].iloc[sorted_idx]
    # Lấy mẫu để tránh quá nhiều điểm
    sample_idx = np.linspace(0, len(x_sorted)-1, 500, dtype=int)
    try:
        y_smooth = savgol_filter(y_sorted.iloc[sample_idx], window_length=51, polyorder=3)
        ax.plot(x_sorted.iloc[sample_idx], y_smooth, 
                'g-', linewidth=2.5, label='Smooth trend', alpha=0.8)
    except:
        pass
    
    # Labels và title
    corr_val = appliances_corr[feature]
    ax.set_xlabel(feature, fontsize=13, fontweight='bold')
    ax.set_ylabel('Appliances (Wh)', fontsize=13, fontweight='bold')
    ax.set_title(f'{feature} vs Appliances\nCorrelation: {corr_val:+.4f}', 
                 fontsize=14, fontweight='bold', pad=10)
    ax.grid(True, alpha=0.3, linewidth=0.8)
    ax.legend(fontsize=10, loc='upper left')
    ax.tick_params(axis='both', labelsize=10)

plt.tight_layout(pad=2.0)
plt.show()

**Phân tích Scatter Plots:**

**1. Đánh giá tính tuyến tính:**
* **Đường đỏ (Linear fit):** Thể hiện mối quan hệ tuyến tính tốt nhất giữa biến độc lập và Appliances.
* **Đường xanh (Smooth trend):** Thể hiện xu hướng thực tế của dữ liệu, có thể phi tuyến.
* **Nếu hai đường trùng nhau:** Mối quan hệ là tuyến tính, Linear Regression phù hợp.
* **Nếu đường xanh cong/gấp khúc:** Mối quan hệ phi tuyến, cần transformation hoặc mô hình phi tuyến.

**2. Các pattern thường gặp:**

**Pattern A - Mối quan hệ tuyến tính mạnh (ví dụ: lights vs Appliances):**
* Các điểm tập trung quanh đường thẳng
* Correlation cao (> 0.5)
* Biến này là predictor tốt cho Linear Regression

**Pattern B - Mối quan hệ phi tuyến (ví dụ: T_out vs Appliances):**
* Đường smooth cong (hình chữ U hoặc chữ S)
* Correlation thấp mặc dù có mối quan hệ rõ ràng
* Cần tạo biến polynomial (T_out², T_out³) hoặc dùng mô hình phi tuyến

**Pattern C - Không có mối quan hệ rõ ràng:**
* Các điểm phân tán ngẫu nhiên
* Correlation gần 0
* Đường smooth gần như nằm ngang
* Biến này có thể không hữu ích cho dự báo

**Pattern D - Heteroscedasticity (Phương sai không đồng nhất):**
* Độ phân tán của điểm thay đổi theo trục X
* Ví dụ: Ở giá trị thấp, điểm tập trung; ở giá trị cao, điểm phân tán
* Vi phạm giả định của Linear Regression, cần transformation hoặc Weighted Least Squares

**3. Phát hiện outliers:**
* Các điểm nằm xa đám đông chính
* Có thể là các sự kiện tiêu thụ năng lượng bất thường
* Cần xem xét có nên giữ lại hay xử lý

## **8. Outlier Detection** 

Phát hiện và phân tích ngoại lai (outliers) là bước quan trọng để đánh giá chất lượng dữ liệu và quyết định chiến lược xử lý phù hợp. Chúng em sẽ áp dụng hai phương pháp thống kê phổ biến (IQR và Z-score) trên các biến quan trọng, sau đó phân tích nguyên nhân và đề xuất hướng xử lý.

### **8.1 Outlier Detection Methods**

Chúng em sẽ áp dụng hai phương pháp phát hiện ngoại lai trên biến mục tiêu `Appliances` và các biến quan trọng khác đã xác định từ phân tích correlation.

In [ ]:
# Chọn các biến quan trọng để phân tích outliers
important_features = ['Appliances', 'lights', 'T1', 'RH_1', 'T_out', 'RH_out', 'Windspeed']

print("Variables selected for outlier analysis:")
for i, feat in enumerate(important_features, 1):
    print(f"{i}. {feat}")

#### **8.1.1 IQR Method (Interquartile Range)**

Phương pháp IQR xác định outliers dựa trên khoảng tứ phân vị. Các giá trị nằm ngoài khoảng `[Q1 - 1.5×IQR, Q3 + 1.5×IQR]` được coi là ngoại lai.

In [ ]:
# Hàm phát hiện outliers bằng IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    return {
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'n_outliers': len(outliers),
        'pct_outliers': (len(outliers) / len(data)) * 100
    }

# Áp dụng IQR method
iqr_results = {}
for feature in important_features:
    iqr_results[feature] = detect_outliers_iqr(df_energy, feature)

# Tạo bảng tổng hợp
iqr_summary = pd.DataFrame({
    'Feature': important_features,
    'Q1': [iqr_results[f]['Q1'] for f in important_features],
    'Q3': [iqr_results[f]['Q3'] for f in important_features],
    'IQR': [iqr_results[f]['IQR'] for f in important_features],
    'Lower Bound': [iqr_results[f]['lower_bound'] for f in important_features],
    'Upper Bound': [iqr_results[f]['upper_bound'] for f in important_features],
    'N Outliers': [iqr_results[f]['n_outliers'] for f in important_features],
    'Outliers %': [iqr_results[f]['pct_outliers'] for f in important_features]
})

print("\nIQR Method Results:")
display(iqr_summary.style
        .format({'Q1': '{:.2f}', 'Q3': '{:.2f}', 'IQR': '{:.2f}',
                 'Lower Bound': '{:.2f}', 'Upper Bound': '{:.2f}',
                 'Outliers %': '{:.2f}%'})
        .background_gradient(cmap='Reds', subset=['Outliers %'])
        .set_caption('Outlier Detection using IQR Method'))

#### **8.1.2 Z-score Method**

Phương pháp Z-score xác định outliers dựa trên độ lệch chuẩn. Các giá trị có `|Z-score| > 3` (tức là cách trung bình hơn 3 độ lệch chuẩn) được coi là ngoại lai.

In [ ]:
# Hàm phát hiện outliers bằng Z-score
def detect_outliers_zscore(data, column, threshold=3):
    mean = data[column].mean()
    std = data[column].std()
    
    z_scores = np.abs((data[column] - mean) / std)
    outliers = data[z_scores > threshold]
    
    return {
        'mean': mean,
        'std': std,
        'threshold': threshold,
        'n_outliers': len(outliers),
        'pct_outliers': (len(outliers) / len(data)) * 100,
        'max_zscore': z_scores.max()
    }

# Áp dụng Z-score method
zscore_results = {}
for feature in important_features:
    zscore_results[feature] = detect_outliers_zscore(df_energy, feature)

# Tạo bảng tổng hợp
zscore_summary = pd.DataFrame({
    'Feature': important_features,
    'Mean': [zscore_results[f]['mean'] for f in important_features],
    'Std': [zscore_results[f]['std'] for f in important_features],
    'Threshold': [zscore_results[f]['threshold'] for f in important_features],
    'Max Z-score': [zscore_results[f]['max_zscore'] for f in important_features],
    'N Outliers': [zscore_results[f]['n_outliers'] for f in important_features],
    'Outliers %': [zscore_results[f]['pct_outliers'] for f in important_features]
})

print("\nZ-score Method Results:")
display(zscore_summary.style
        .format({'Mean': '{:.2f}', 'Std': '{:.2f}', 'Max Z-score': '{:.2f}',
                 'Outliers %': '{:.2f}%'})
        .background_gradient(cmap='Reds', subset=['Outliers %'])
        .set_caption('Outlier Detection using Z-score Method (threshold=3)'))

In [ ]:
# So sánh hai phương pháp
comparison = pd.DataFrame({
    'Feature': important_features,
    'IQR Outliers': [iqr_results[f]['n_outliers'] for f in important_features],
    'IQR %': [iqr_results[f]['pct_outliers'] for f in important_features],
    'Z-score Outliers': [zscore_results[f]['n_outliers'] for f in important_features],
    'Z-score %': [zscore_results[f]['pct_outliers'] for f in important_features],
    'Difference': [abs(iqr_results[f]['n_outliers'] - zscore_results[f]['n_outliers']) 
                   for f in important_features]
})

print("\nComparison of IQR vs Z-score Methods:")
display(comparison.style
        .format({'IQR %': '{:.2f}%', 'Z-score %': '{:.2f}%'})
        .background_gradient(cmap='YlOrRd', subset=['IQR %', 'Z-score %'])
        .set_caption('Comparison: IQR vs Z-score Outlier Detection'))

### **8.2 Visualization of Outliers**

Để hiểu rõ hơn về phân bố và vị trí của các outliers, chúng em vẽ boxplots cho các biến quan trọng với các ngưỡng IQR được đánh dấu rõ ràng.

In [ ]:
# Vẽ boxplots cho các biến quan trọng
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for idx, feature in enumerate(important_features):
    ax = axes[idx]
    
    # Boxplot
    bp = ax.boxplot(df_energy[feature], vert=True, patch_artist=True, widths=0.6,
                      boxprops=dict(facecolor='lightblue', color='blue', linewidth=2),
                      medianprops=dict(color='red', linewidth=3),
                      whiskerprops=dict(color='blue', linewidth=2),
                      capprops=dict(color='blue', linewidth=2),
                      flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
    
    # Thêm thông tin IQR
    iqr_info = iqr_results[feature]
    ax.axhline(y=iqr_info['lower_bound'], color='green', linestyle='--', linewidth=2, 
               label=f'Lower: {iqr_info["lower_bound"]:.1f}')
    ax.axhline(y=iqr_info['upper_bound'], color='orange', linestyle='--', linewidth=2, 
               label=f'Upper: {iqr_info["upper_bound"]:.1f}')
    
    # Labels và title
    ax.set_ylabel(feature, fontsize=13, fontweight='bold')
    ax.set_title(f'{feature}\nOutliers: {iqr_info["n_outliers"]} ({iqr_info["pct_outliers"]:.2f}%)', 
                 fontsize=14, fontweight='bold', pad=10)
    ax.grid(True, alpha=0.3, axis='y', linewidth=0.8)
    ax.legend(fontsize=10, loc='upper right')
    ax.tick_params(axis='both', labelsize=11)

# Ẩn subplot thừa
axes[7].axis('off')

plt.tight_layout(pad=2.0)
plt.show()

### **8.3 Root Cause Analysis & Treatment Recommendations**

**Phân tích nguyên nhân xuất hiện outliers:**

**1. Outliers trong biến `Appliances` (Biến mục tiêu):**

* **Nguyên nhân:** Hành vi sử dụng thực tế khi nhiều thiết bị công suất lớn hoạt động đồng thời (lò nướng + máy giặt + máy sấy + điều hòa). Các giá trị cao (800-1080 Wh) thường xảy ra vào cuối tuần hoặc buổi tối.
* **Tính hợp lý:** Tổng công suất các thiết bị trong nhà có thể lên đến 10,000W, do đó mức tiêu thụ 1000 Wh trong 10 phút là hoàn toàn khả thi.
* **Kết luận:** Đây là outliers **CÓ Ý NGHĨA**, phản ánh hành vi thực tế cần dự báo.

**2. Outliers trong các biến môi trường (T1-T9, RH_1-RH_9):**

* **Nguyên nhân:** Biến động tự nhiên do mở cửa sổ, nấu ăn, tắm rửa, bật/tắt điều hòa.
* **Độ tin cậy:** Cảm biến ZigBee có độ chính xác ±0.5°C và ±3% RH, các giá trị cực đoan vẫn nằm trong phạm vi đo lường hợp lệ.
* **Kết luận:** Outliers **CÓ Ý NGHĨA**, phản ánh các sự kiện thực tế trong nhà.

**3. Outliers trong biến thời tiết (T_out, RH_out, Windspeed):**

* **Nguyên nhân:** Biến động thời tiết tự nhiên (bão, sóng lạnh, nắng nóng).
* **Độ tin cậy:** Dữ liệu từ trạm khí tượng chính thức, đã qua kiểm định.
* **Kết luận:** Outliers **CÓ Ý NGHĨA**, cần giữ lại để mô hình học được ảnh hưởng của thời tiết cực đoan.


**Đề xuất hướng xử lý cho Linear Regression:**

Dựa trên phân tích nguyên nhân và yêu cầu của bài toán Regression với Linear Regression, chúng em đề xuất chiến lược xử lý outliers như sau:

Giữ nguyên tất cả outliers.

**Lý do chi tiết:**

**1. Outliers phản ánh hành vi thực tế quan trọng:**
* Các giá trị tiêu thụ cao (outliers) là chính xác những trường hợp mà mô hình cần học để dự báo
* Trong thực tế triển khai, hệ thống sẽ gặp các tình huống tiêu thụ cao này và cần dự đoán chính xác
* Loại bỏ outliers = Mô hình sẽ under-predict nghiêm trọng khi gặp các trường hợp tiêu thụ cao

**2. Phù hợp với các phương pháp xử lý khác:**
* **Power Transformation (Yeo-Johnson)** đã được áp dụng trong phần Feature Scaling (Section 9.4) sẽ tự động xử lý vấn đề phân phối lệch và giảm ảnh hưởng của outliers mà không cần loại bỏ chúng
* Transformation này "nắn" phân phối về dạng gần chuẩn hơn, giúp Linear Regression hoạt động tốt hơn với outliers
* Kết hợp với StandardScaler sau đó sẽ đưa tất cả biến về cùng thang đo

**3. Tương thích với Linear Regression:**
* Mặc dù Linear Regression nhạy cảm với outliers, nhưng với Power Transformation, vấn đề này đã được giảm thiểu đáng kể
* Các giả định của Linear Regression (linearity, homoscedasticity, normality of residuals) sẽ được kiểm tra sau khi train mô hình
* Nếu vẫn vi phạm nghiêm trọng, có thể chuyển sang Robust Regression (Huber, RANSAC) hoặc các mô hình phi tuyến

**4. Bảo toàn tính toàn vẹn dữ liệu:**
* Giữ nguyên 100% dữ liệu (19,735 quan sát)
* Không mất thông tin về các pattern quan trọng
* Tránh bias trong quá trình training

## **9. Data Preprocessing & Feature Engineering**

### **9.1 Missing Data**

Phân tích dữ liệu thiếu là bước then chốt để đánh giá mức độ hoàn thiện và độ tin cậy của bộ dữ liệu. Đối với dữ liệu chuỗi thời gian, mục tiêu của phần này là rà soát toàn diện nhằm xác định xem có bất kỳ sự đứt gãy thông tin nào từ các thiết bị đo lường (như đồng hồ điện, mạng lưới cảm biến) hay không. Việc xác nhận một bộ dữ liệu đầy đủ sẽ giúp chúng ta đảm bảo không làm sai lệch các quy luật vật lý và hành vi tiêu thụ năng lượng nguyên bản của ngôi nhà.

#### **Missing Summary Table**

Chúng ta bắt đầu bằng cái nhìn tổng quan định lượng về tỷ lệ thiếu của từng biến số.

In [ ]:
# Tạo bảng thống kê dữ liệu thiếu
missing_counts = df_energy.isnull().sum()
total_cells = df_energy.size
total_missing = missing_counts.sum()

missing_df = pd.DataFrame({
    'Feature': df_energy.columns,
    'Missing Count': missing_counts.values,
    'Missing %': (missing_counts.values / len(df_energy)) * 100
}).sort_values(by='Missing Count', ascending=False)

cols_with_missing = (missing_df["Missing Count"] > 0).sum()
overall_missing_rate = (total_missing / total_cells) * 100

print("Dataset Missing Overview:")
print(f"- Total Missing Values    : {total_missing:,.0f}")
print(f"- Overall Missing Rate    : {overall_missing_rate:.2f}%")
print(f"- Columns With Missing    : {cols_with_missing} / {len(df_energy.columns)}\n")

print("Detailed Missingness by Feature (Descending):")

# Hiển thị bảng định dạng với thanh màu
display(
    missing_df.style
        .format({"Missing %": "{:.2f}%"})
        .bar(subset=["Missing %"], color="#FF6B6B", vmin=0, vmax=100)
        .background_gradient(subset=["Missing Count"], cmap="Reds")
)


#### **Nhận xét & Chiến lược xử lý**

Như kết quả thống kê ở trên, bộ dữ liệu **Appliances Energy Prediction** đạt mức độ hoàn thiện lý tưởng với **0% dữ liệu thiếu** trên toàn bộ 29 không gian đặc trưng. Cả 19.735 quan sát đều có đầy đủ thông tin từ biến thời gian, biến mục tiêu cho đến hệ thống cảm biến môi trường.

**Nguyên nhân dữ liệu sạch:**
Sự trọn vẹn này có được là nhờ tính chất tự động hóa của hệ thống thu thập dữ liệu IoT (mạng lưới cảm biến không dây ZigBee trong nhà và trạm khí tượng tự động). Cứ mỗi 10 phút, hệ thống sẽ tự động đồng bộ và nội suy các thông số mà không phụ thuộc vào sự ghi chép thủ công của con người.

**Chiến lược xử lý:**
* **Bỏ qua bước Imputation:** Do không có giá trị `NaN`, chúng ta hoàn toàn không cần áp dụng các kỹ thuật điền khuyết (như Mean, Median hay MICE).
* **Tiến thẳng đến Feature Engineering:** Dữ liệu đã sẵn sàng để chuyển sang các bước biến đổi đặc trưng và chia tập (Train/Test Split).


### **9.2 Feature Engineering**

Tiếp theo, nhóm sẽ tiến hành trích xuất các đặc trưng mới từ dữ liệu gốc nhằm giúp thuật toán học máy bắt được các chu kỳ sinh hoạt của con người và quy luật truyền nhiệt của ngôi nhà một cách rõ ràng nhất.

In [ ]:
# Tạo bản sao để tránh làm thay đổi dữ liệu gốc ban đầu
df_fe = df_energy.copy()

# Chuyển cột date sang định dạng datetime
df_fe['date'] = pd.to_datetime(df_fe['date'])

# Bóc tách các thành phần thời gian
df_fe['Month'] = df_fe['date'].dt.month
df_fe['DayOfWeek'] = df_fe['date'].dt.dayofweek # Thứ 2 = 0, Thứ 3 = 1, ... , Chủ nhật = 6
df_fe['Hour'] = df_fe['date'].dt.hour
df_fe['Minute'] = df_fe['date'].dt.minute

# Tạo cờ cuối tuần (Thứ 7, CN)
df_fe['Is_Weekend'] = df_fe['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Tạo cờ giờ hành chính (8h sáng đến 17h chiều, và không phải cuối tuần)
df_fe['Is_Business_Hour'] = ((df_fe['Hour'] >= 8) & (df_fe['Hour'] < 17) & (df_fe['Is_Weekend'] == 0)).astype(int)

# Tính nhiệt độ trung bình các phòng trong nhà (Từ T1 đến T9, bỏ T6 vì T6 là mặt ngoài Bắc ngôi nhà)
indoor_temp_cols = ['T1', 'T2', 'T3', 'T4', 'T5', 'T7', 'T8', 'T9']
df_fe['T_indoor_avg'] = df_fe[indoor_temp_cols].mean(axis=1)

# Tính độ chênh lệch nhiệt độ trong và ngoài nhà
df_fe['Temp_Diff'] = df_fe['T_indoor_avg'] - df_fe['T_out']

cols_to_show = ['date', 'Month', 'DayOfWeek', 'Hour', 'Minute','Is_Weekend', 'Is_Business_Hour', 'T_indoor_avg', 'Temp_Diff']
print("\n10 dòng dữ liệu ngẫu nhiên của các biến mới tạo")
display(df_fe[cols_to_show].sample(n=10, random_state=42).sort_index())

# Drop cột date sau khi kiểm tra xong
df_fe = df_fe.drop(columns=['date'], errors='ignore')

Xuất phát từ đặc thù của mạng lưới cảm biến và tính chu kỳ trong hành vi sinh hoạt, quá trình kỹ nghệ đặc trưng được thiết kế xoay quanh hai nhóm biến số trọng tâm:

**1. Trích xuất đặc trưng thời gian:**
* **Vấn đề:** Biến `date` là chuỗi thời gian liên tục, các thuật toán Hồi quy hay Cây quyết định không thể đọc và tính toán trực tiếp trên định dạng này.
* **Giải pháp & Lý do lựa chọn:** Phân rã `date` thành các đặc trưng rời rạc để mô phỏng chính xác nhịp điệu sinh hoạt của gia đình:
  * `Month` (Tháng trong năm): Giúp thuật toán nhận biết sự thay đổi của thời tiết theo tháng, từ đó dự báo xu hướng tiêu thụ năng lượng thiết bị làm mát/sưởi ấm.
  * `DayOfWeek` (Ngày trong tuần) & `Is_Weekend` (Cuối tuần): Tần suất có mặt ở nhà và nhu cầu nấu nướng, giặt giũ vào ngày nghỉ (Thứ 7, Chủ nhật) thường cao hơn hẳn các ngày đi làm trong tuần.
  * `Hour` (Giờ trong ngày): Là yếu tố quan trọng nhất. Hành vi dùng điện có sự phân hóa rõ rệt theo giờ (ban đêm dùng nhiều thiết bị hơn ban ngày).
  * `Minute` (Phút trong giờ): Do dữ liệu được ghi nhận với tần suất 10 phút/lần, việc bổ sung biến Phút giúp mô hình nắm bắt được chu kì hoạt động của thiết bị (ví dụ: chu kỳ đóng/ngắt tự động của tủ lạnh, máy sưởi) – những biến động ngắn hạn mà đặc trưng Giờ không đủ độ phân giải để phản ánh.
  * `Is_Business_Hour` (Giờ hành chính): Xác định khung giờ làm việc (từ 8h sáng đến 5h chiều) của các ngày trong tuần. Trong khoảng thời gian này, nhà thường trống nên mức tiêu thụ năng lượng sẽ tụt xuống mức nền.
 
**2. Tạo đặc trưng nhiệt động lực học:**
* **Vấn đề:** Dữ liệu cung cấp nhiệt độ của 9 phòng khác nhau, nhưng năng lượng tổng (`Appliances`) lại phục vụ cho cả ngôi nhà.
* **Giải pháp & Lý do lựa chọn:** 
  * `T_indoor_avg` (Nhiệt độ trung bình trong nhà): Gộp nhiệt độ các phòng (trừ mặt ngoài T6) để tạo ra một đại lượng đại diện cho mức nhiệt năng chung của toàn bộ không gian sống.
  * `Temp_Diff` (Độ chênh lệch nhiệt độ): Bằng Nhiệt độ trung bình trong nhà trừ đi Nhiệt độ ngoài trời (`T_out`). Theo nguyên lý nhiệt động lực học, mức độ chênh lệch nhiệt độ giữa trong và ngoài vỏ công trình càng lớn thì hệ thống điều hòa/sưởi ấm càng phải hoạt động với công suất cao để bù đắp.

#### **Cyclical Feature Encoding**

In [ ]:
# 1. Mã hóa cho Tháng (Chu kỳ 12)
df_fe['Month_sin'] = np.sin(2 * np.pi * df_fe['Month'] / 12)
df_fe['Month_cos'] = np.cos(2 * np.pi * df_fe['Month'] / 12)

# 2. Mã hóa cho Ngày trong tuần (Chu kỳ 7)
df_fe['DayOfWeek_sin'] = np.sin(2 * np.pi * df_fe['DayOfWeek'] / 7)
df_fe['DayOfWeek_cos'] = np.cos(2 * np.pi * df_fe['DayOfWeek'] / 7)

# 3. Mã hóa cho Giờ (Chu kỳ 24)
df_fe['Hour_sin'] = np.sin(2 * np.pi * df_fe['Hour'] / 24)
df_fe['Hour_cos'] = np.cos(2 * np.pi * df_fe['Hour'] / 24)

# 4. Mã hóa cho Phút (Chu kỳ 60)
df_fe['Minute_sin'] = np.sin(2 * np.pi * df_fe['Minute'] / 60)
df_fe['Minute_cos'] = np.cos(2 * np.pi * df_fe['Minute'] / 60)

# Lấy 10 dòng ngẫu nhiên và sắp xếp theo thời gian để xem kết quả
cols_to_show = [
    'Month', 'DayOfWeek', 'Hour', 'Minute',
    'Month_sin', 'Month_cos', 
    'DayOfWeek_sin', 'DayOfWeek_cos', 
    'Hour_sin', 'Hour_cos', 
    'Minute_sin', 'Minute_cos'
]
display(df_fe[cols_to_show].sample(n=10, random_state=42).sort_index())

# Xóa bỏ các cột số nguyên gốc để tránh đa cộng tuyến
df_fe = df_fe.drop(columns=['Month', 'DayOfWeek', 'Hour', 'Minute'])


Các biến thời gian như Giờ trong ngày (`Hour`), Phút trong giờ (`Minute`), Ngày trong tuần (`DayOfWeek`), và Tháng trong năm (`Month`) có tính chất chu kỳ lặp lại. Nếu giữ nguyên dưới dạng số nguyên, mô hình sẽ hiểu sai khoảng cách vật lý. Ví dụ: Sự chênh lệch giữa 23h và 0h sẽ bị mô hình tính toán là 23 đơn vị khoảng cách, trong khi thực tế chúng chỉ cách nhau 1 giờ.

Để giải quyết vấn đề này, nhóm quyết định sử dụng **Mã hóa Sin/Cos**. Bằng cách chiếu các mốc thời gian lên một hệ tọa độ đường tròn lượng giác, điểm cuối chu kỳ sẽ được kết nối mượt mà với điểm đầu chu kỳ. 

### **9.3 Data Splitting Strategy**

* **Tỷ lệ chia:** Dữ liệu được chia thành 3 tập Train / Validation / Test với tỷ lệ tương ứng là **70% / 15% / 15%**.
* **Đặc thù chuỗi thời gian:** Vì dữ liệu phụ thuộc rất lớn vào tính tuần tự của thời gian (nhiệt độ phút này phụ thuộc phút trước), nhóm **không xáo trộn ngẫu nhiên (`shuffle=False`)**. Dữ liệu sẽ được cắt theo đúng trục thời gian tuyến tính để mô hình học từ quá khứ (tháng 1 đến tháng 4) và dự báo cho tương lai (tháng 5), qua đó ngăn chặn rò rỉ dữ liệu tương lai vào tập huấn luyện.

In [ ]:
# Tách biến độc lập (X) và biến mục tiêu (y)
X = df_fe.drop(columns=['Appliances'])
y = df_fe['Appliances']

# Chia tập dữ liệu với tỷ lệ 70/15/15 (Không dùng shuffle vì là Time-Series)
# Lần cắt 1: Lấy 85% cho (Train + Val) và 15% cho Test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)

# Lần cắt 2: Trong 85% đó, chia lấy phần Validation (chiếm 15%)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(0.15/0.85), shuffle=False)

print("Kích thước các tập dữ liệu:")
print(f"- Tập Train      : {X_train.shape[0]} dòng ({len(X_train)/len(df_fe)*100:.0f}%)")
print(f"- Tập Validation : {X_val.shape[0]} dòng ({len(X_val)/len(df_fe)*100:.0f}%)")
print(f"- Tập Test       : {X_test.shape[0]} dòng ({len(X_test)/len(df_fe)*100:.0f}%)")

### **9.4 Feature Scaling**

**Đánh giá phân bố Skewness & Kurtosis để lựa chọn thuật toán Scaling**

Trước khi tiến hành chuẩn hóa dữ liệu, nhóm không lựa chọn thuật toán Scaling một cách cảm tính mà dựa trên cơ sở toán học về hình dáng phân bố của các đặc trưng. Nhóm tiến hành đo lường hai chỉ số thống kê quan trọng:
1. **Độ lệch (Skewness):** Đánh giá tính đối xứng của dữ liệu. Nếu giá trị cách xa 0, dữ liệu bị lệch (trái hoặc phải) rất nặng.
2. **Bề dày đuôi (Kurtosis):** Đánh giá mức độ tập trung của ngoại lai ở phần đuôi. Phân bố chuẩn có Kurtosis bằng 0. Giá trị Kurtosis càng lớn dương chứng tỏ phần đuôi chứa càng nhiều ngoại lai cực đoan.

Mục tiêu của bước này là kiểm tra xem các biến có vi phạm giả định phân phối chuẩn hay không, từ đó có căn cứ loại trừ các thuật toán Scaling không phù hợp.

In [ ]:
# 1. Lọc các biến định lượng liên tục 
# bỏ qua các cờ nhị phân
cols_to_exclude = ['Is_Weekend', 'Is_Business_Hour']

# tự động chọn các cột số, không nằm trong danh sách loại trừ, và không có đuôi '_sin' hay '_cos' 
num_features = [col for col in df_fe.columns 
                if col not in cols_to_exclude 
                and not col.endswith('_sin') 
                and not col.endswith('_cos')
                and df_fe[col].dtype in ['float64', 'int64']]

# 2. Tính toán skewness và kurtosis
stats_df = pd.DataFrame({
    'Feature': num_features,
    'Skewness': df_fe[num_features].skew().values,
    'Kurtosis': df_fe[num_features].kurt().values 
}).reset_index(drop=True)

# 3. Phân loại mức độ lệch
def skew_condition(x):
    if abs(x) < 0.5: return "~ chuẩn"
    elif abs(x) < 1: return "lệch vừa"
    else: return "lệch nặng"

stats_df['Đánh giá phân bố'] = stats_df['Skewness'].apply(skew_condition)

# 4. Sắp xếp theo độ lệch giảm dần
stats_df = stats_df.sort_values(by='Skewness', key=abs, ascending=False)

print("Bảng thống kê skewness & kurtosis của từng đặc trưng")
display(stats_df.style.background_gradient(cmap='Oranges', subset=['Skewness', 'Kurtosis']))

**Nhận xét và chiến lược xử lý:**

Từ bảng thống kê phân bố trên, nhóm rút ra các kết luận sau:

* **1. Tiêu chí chọn lọc đặc trưng phân tích:** Trước khi đo lường Skewness và Kurtosis, nhóm loại trừ các cờ nhị phân (trạng thái 0/1 như `Is_Weekend`) và các biến chu kỳ lượng giác (`_sin`, `_cos`) vốn ở biên độ [-1, 1]. Lý do là các phép đo hình dáng phân bố chỉ có ý nghĩa thống kê đối với các **biến định lượng vật lý liên tục** (như nhiệt độ, độ ẩm, điện năng). Việc cố gắng nắn chỉnh phân bố trên các biến phân loại hoặc biến trạng thái là phi logic.

* **2. Hệ quả đến việc lựa chọn thuật toán chuẩn hóa:**
  * Việc sử dụng các thuật toán Scale đơn thuần ngay từ đầu là không an toàn. Đặc biệt là **`MinMaxScaler`**, các ngoại lai ở đuôi sẽ đẩy giá trị Max lên quá cao, vô tình ép 99% dữ liệu sinh hoạt bình thường về sát mốc 0. Điều này làm mất phương sai và phá hỏng các hàm phạt của mô hình.
  * Việc dùng **`StandardScaler`** trực tiếp lên dữ liệu gốc cũng sẽ bị khối lượng ngoại lai kéo lệch tham số mean. Tuy nhiên, nếu dữ liệu được xử lý triệt để vấn đề phân bố lệch trước đó, `StandardScaler` lại là sự lựa chọn tối ưu nhất cho các mô hình hồi quy tuyến tính.

* **3. Quyết định cuối cùng:** Dựa trên các minh chứng thống kê này, nhóm quyết định kết hợp **`Power Transformer` (Yeo-Johnson)** và **`StandardScaler`** để chuẩn hóa dữ liệu:
  * **Bước 1 - Nắn hình dáng:** Thuật toán Yeo-Johnson sẽ tự động dò tìm số mũ tối ưu để bẻ cong các phân bố bị lệch về dạng tiệm cận hình chuông đối xứng. Bước này giúp giải quyết triệt để vấn đề Skewness.
  * **Bước 2 - Chuẩn hóa thang đo:** Sau khi dữ liệu đã có hình dáng phân bố chuẩn từ Bước 1, nó thỏa mãn điều kiện lý tưởng của StandardScaler. Thuật toán này sẽ tự tin đưa toàn bộ các đặc trưng về chung một hệ quy chiếu (Trung bình = 0, Độ lệch chuẩn = 1), giúp các thuật toán như Ridge, Lasso hay ElasticNet hội tụ nhanh và dự báo chính xác.

In [ ]:
# Tách riêng các nhóm biến
cols_no_scale = [col for col in X_train.columns if col.startswith('Is_') or col.endswith('_sin') or col.endswith('_cos')]
cols_continuous = [col for col in X_train.columns if col not in cols_no_scale]

# Tính toán skewness trên tập train để tự động lọc ra các cột có |skew| > 0.5
skew_series = X_train[cols_continuous].skew()
cols_skewed = skew_series[abs(skew_series) > 0.5].index.tolist()
cols_normal = [col for col in cols_continuous if col not in cols_skewed]

# Thiết lập bộ xử lý riêng biệt
# nhóm lệch nặng: power transformer -> standard scaler
# nhóm còn lại: chỉ standard scaler
preprocessor = ColumnTransformer(
    transformers=[
        ('skewed_pipe', Pipeline([
            ('power', PowerTransformer(method='yeo-johnson', standardize=False)),
            ('scaler', StandardScaler())
        ]), cols_skewed),
        
        ('normal_pipe', StandardScaler(), cols_normal),
        
        ('pass', 'passthrough', cols_no_scale) # giữ nguyên các cột nhị phân/chu kỳ
    ]
)

# Thực thi biến đổi
# ta cần lấy lại danh sách tên cột theo đúng thứ tự mà transformer trả ra
new_col_order = cols_skewed + cols_normal + cols_no_scale

X_train_scaled = pd.DataFrame(preprocessor.fit_transform(X_train), columns=new_col_order, index=X_train.index)
X_val_scaled = pd.DataFrame(preprocessor.transform(X_val), columns=new_col_order, index=X_val.index)
X_test_scaled = pd.DataFrame(preprocessor.transform(X_test), columns=new_col_order, index=X_test.index)


**Save Processed Data**

In [ ]:
# Lưu tập Train
df_train_processed = X_train_scaled.copy()
df_train_processed['Appliances'] = y_train.values
df_train_processed.to_csv('../../data/processed/Energy_Use_train.csv', index=False)

# Lưu tập Validation
df_val_processed = X_val_scaled.copy()
df_val_processed['Appliances'] = y_val.values
df_val_processed.to_csv('../../data/processed/Energy_Use_val.csv', index=False)

# Lưu tập Test
df_test_processed = X_test_scaled.copy()
df_test_processed['Appliances'] = y_test.values
df_test_processed.to_csv('../../data/processed/Energy_Use_test.csv', index=False)

# Hiển thị 5 dòng ngẫu nhiên của tập train để kiểm tra
display(X_train_scaled.sample(5, random_state=42))

---